In [2]:
from __future__ import annotations

import re
from difflib import SequenceMatcher

def str_to_bool(value, threshold=0.7):
    """
    Convert a string representation of a boolean to an actual boolean using
    edit distance to match against known true/false synonyms.
    
    Args:
        value: The input value to convert (string, bool, etc.)
        threshold: Minimum similarity score (0-1) to consider a match
        
    Returns:
        bool: The boolean interpretation of the input
    """
    
    # Handle non-string types
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    if isinstance(value, (int, float)):
        return bool(value)
    
    # Convert to string and normalize
    text = str(value).lower().strip()
    
    # Clean the text to remove punctuation and extra whitespace
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # If empty after cleaning
    if not text:
        return False
    
    # Define synonym lists
    true_synonyms = [
        'true', 'yes', 'y', 'yeah', 'yep', 'correct', 'right',
        'affirmative', '1', 'ok', 'okay', 'sure', 'positive',
        'indeed', 'absolutely', 'definitely', 'certainly'
    ]
    
    false_synonyms = [
        'false', 'no', 'n', 'nope', 'nah', 'incorrect', 'wrong',
        'negative', '0', 'zero', 'never', 'not', 'none', 'deny'
    ]
    
    # Get the first word (most likely to contain the boolean value)
    first_word = text.split()[0] if text.split() else text
    
    # Calculate similarity scores
    max_true_score = 0
    max_false_score = 0
    
    for synonym in true_synonyms:
        score = SequenceMatcher(None, first_word, synonym).ratio()
        if score > max_true_score:
            max_true_score = score
    
    for synonym in false_synonyms:
        score = SequenceMatcher(None, first_word, synonym).ratio()
        if score > max_false_score:
            max_false_score = score
    
    # Determine result based on scores and threshold
    if max_true_score > threshold or max_false_score > threshold:
        # If both exceed threshold, use the better match
        if max_true_score > max_false_score:
            return True
        else:
            return False
    elif max_true_score > max_false_score:
        # If neither exceeds threshold but true score is higher
        return True
    else:
        # Default to false
        return False

In [3]:
import litellm
import dotenv

from functools import partial

MODEL_VALUES = dotenv.dotenv_values()

completion = partial(litellm.completion, **MODEL_VALUES)

In [4]:
from analysis.models.data import Data

with open("../data/data.json", "r") as f:
    data = Data.model_validate_json(f.read())

/Users/calvin/all-hands/agent-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import json
import time
from typing import Any
from pydantic import BaseModel

class Feature(BaseModel):
    identifier: str
    description: str

    @property
    def to_tool_description_field(self) -> dict[str, Any]:
        return {
            "type": "boolean",
            "description": self.description,
        }

class EmbeddingDimension(BaseModel):
    feature_id: str
    result: bool
    
class FeatureEmbedding(BaseModel):
    featurizer_id: str
    issue_id: str
    issue_description: str
    dimensions: list[EmbeddingDimension]
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    response_latency: float | None = None

    @property
    def cost(self) -> float:
        input_cost, output_cost = litellm.cost_calculator.cost_per_token(
            model=MODEL_VALUES["model"].removeprefix("litellm_proxy/"),
            prompt_tokens=self.prompt_tokens if self.prompt_tokens else 0,
            completion_tokens=self.completion_tokens if self.completion_tokens else 0,
        )
        return input_cost + output_cost

    def to_row(self) -> dict[str, Any]:
        return {
            "featurizer_id": self.featurizer_id,
            "issue_id": self.issue_id,
            "response_latency": self.response_latency,
            "cost": self.cost,
            **{dimension.feature_id: dimension.result for dimension in self.dimensions},
        }

class Featurizer(BaseModel):
    identifier: str
    system_prompt: str
    message_prefix: str
    features: list[Feature]

    def system_message(self) -> dict[str, Any]:
        return {
            "role": "system",
            "content": self.system_prompt,
        }
    
    def user_message(self, issue_description: str) -> dict[str, Any]:
        return {
            "role": "user",
            "content": f"{self.message_prefix}{issue_description}",
        }
    
    @property
    def tool_choice(self) -> dict[str, Any]:
        return {
            "type": "function",
            "function": {"name": f"call_featurizer_{self.identifier}"},
        }

    @property
    def tool_description(self) -> dict[str, Any]:
        return {
            "type": "function",
            "function": {
                "name": f"call_featurizer_{self.identifier}",
                "description": "Record the features present in the issue.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        feature.identifier: feature.to_tool_description_field
                        for feature in self.features
                    },
                },
            },
        }

    def embed(self, issue_id: str, issue_description: str, temperature: float = 1.0) -> FeatureEmbedding:
        start_time = time.time()
        response = completion(
            messages=[self.system_message(), self.user_message(issue_description)],
            tools=[self.tool_description],
            tool_choice=self.tool_choice,
            temperature=temperature,
        )
        stop_time = time.time()

        response_latency = stop_time - start_time
        features = response.choices[0].message.tool_calls[0].function.arguments
        embedding = json.loads(features)

        return FeatureEmbedding(
            featurizer_id=self.identifier,
            issue_id=issue_id,
            issue_description=issue_description,
            dimensions=[
                EmbeddingDimension(
                    feature_id=feature_id,
                    result=result,
                )
                for feature_id, result in embedding.items()
            ],
            response_latency=response_latency,
            prompt_tokens=response.usage.prompt_tokens,
            completion_tokens=response.usage.completion_tokens,
        )


In [6]:
negative_features = [
    Feature(
        identifier="database_specific",
        description="The issue involves database-specific behaviors or differences between database backends"
    ),
    Feature(
        identifier="parsing_regex",
        description="The issue involves parsing, tokenization, or regular expressions"
    ),
    Feature(
        identifier="config_special_chars",
        description="The issue involves configuration or settings handling with special characters or syntax"
    ),
    Feature(
        identifier="type_coercion",
        description="The issue requires understanding type coercion or handling across system boundaries"
    ),
    Feature(
        identifier="math_edge_cases",
        description="The issue involves mathematical or computational edge cases"
    ),
    Feature(
        identifier="cross_platform",
        description="The issue requires knowledge of implementation differences across platforms or versions"
    ),
    Feature(
        identifier="undocumented_assumptions",
        description="The issue involves undocumented assumptions or behaviors in the codebase"
    ),
    Feature(
        identifier="environment_dependent",
        description="The issue appears in some environments but not others"
    ),
    Feature(
        identifier="encoding_i18n",
        description="The issue involves character encoding, internationalization, or locale-specific behavior"
    ),
    Feature(
        identifier="component_interactions",
        description="The issue requires understanding subtle interactions between components"
    ),
    Feature(
        identifier="resource_optimization",
        description="The issue involves memory management, resource handling, or performance optimization"
    ),
    Feature(
        identifier="complex_data_formats",
        description="The issue involves parsing or generating complex data formats (JSON, XML, etc.)"
    ),
]

positive_features = [
    Feature(
        identifier="clear_isolated_cause",
        description="The issue has a clear, isolated cause with explicit error messages"
    ),
    Feature(
        identifier="simple_edge_case",
        description="The solution likely involves adding a missing check or handling a clearly defined edge case"
    ),
    Feature(
        identifier="reproducible",
        description="The issue can be reproduced consistently with the given steps"
    ),
    Feature(
        identifier="localized_fix",
        description="The fix is likely confined to a single function or small section of code"
    ),
    Feature(
        identifier="single_component",
        description="The issue doesn't involve multiple interacting components or systems"
    ),
    Feature(
        identifier="standard_patterns",
        description="The code in question follows standard patterns documented in the framework"
    ),
    Feature(
        identifier="no_deep_implementation",
        description="The solution doesn't require deep understanding of internal implementation details"
    ),
]

system_prompt="""
# GitHub Issue AI Solvability Classifier

You are a classifier that evaluates whether a GitHub issue can be successfully resolved by an AI software engineering agent. Your job is to analyze the issue description and determine if an AI can handle it effectively or if human expertise is required.

## Classification Guidelines

Analyze each GitHub issue carefully and classify it as either:
- **AI-Solvable** (True): The issue can likely be resolved correctly by an AI agent
- **Human-Required** (False): The issue requires human expertise and understanding

## Classification Logic

When evaluating issues, consider that AI agents have general knowledge of standard programming concepts, library APIs, and common frameworks, but struggle with nuanced technical understanding, system-specific knowledge, and complex interactions.

Remember that it's better to be cautious and classify an issue as requiring human expertise than to overestimate AI capabilities. When in doubt, classify as "Human-Required" (False).

## Input Format
You'll receive GitHub issue descriptions with details about the problem, error messages, and reproduction steps.

## Output Format
Provide a boolean classification (True/False) and a brief explanation of your reasoning.

AI-Solvable = True
Human-Required = False
"""

featurizer = Featurizer(
    identifier="baseline",
    system_prompt=system_prompt,
    message_prefix="Github issue description: ",
    features=negative_features + positive_features,
)

In [7]:
from analysis.models.swe_bench import Instance
import pandas as pd

def featurize(instances: list[Instance], featurizer: Featurizer) -> pd.DataFrame:
    """
    Featurize a list of instances using the provided featurizer.
    
    Args:
        instances: List of instances to featurize
        featurizer: The featurizer to use for embedding
    
    Returns:
        pd.DataFrame: DataFrame containing the featurized data
    """
    results = []
    for instance in instances:
        result = featurizer.embed(instance.instance_id, instance.problem_statement)
        results.append(result.to_row())
    
    return pd.DataFrame(results)

try:
    df = pd.read_csv("featurizer_baseline.csv")
    df["resolved"] = df["issue_id"].apply(data.systems["20250203_openhands_4x_scaled"].results.is_resolved)
except FileNotFoundError:
    df = featurize(data.dataset.instances, featurizer)
    df.to_csv("featurizer_baseline.csv", index=False)

In [8]:
from sklearn.model_selection import train_test_split

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df[[feature.identifier for feature in featurizer.features]], df['resolved'], test_size=0.2, random_state=42
)

In [9]:
# Compute ROC curve against a particular system in the data
from sklearn.metrics import roc_curve
import altair as alt

def roc(y_scores: pd.Series, y_true: pd.Series) -> alt.Chart:
    """
    Compute the ROC curve for a given system.
    """
    df = pd.DataFrame({
        "y_scores": y_scores,
        "y_true": y_true,
    })

    fpr, tpr, thresholds = roc_curve(df['y_true'], df['y_scores'])
    curve = pd.DataFrame({
        "fpr": fpr,
        "tpr": tpr,
        "threshold": thresholds
    })

    p_ratio = df['y_true'].sum() / len(df)
    curve["accuracy"] = p_ratio * curve["tpr"] + (1 - p_ratio) * (1 - curve["fpr"])

    chart = alt.Chart(curve).mark_line().encode(
        x=alt.X("fpr", title="False Positive Rate"),
        y=alt.Y("tpr", title="True Positive Rate"),
        tooltip=["fpr", "tpr", "threshold", "accuracy"],
    )

    # Compute the threshold/accuracy pair with the highest accuracy
    best_threshold = curve.loc[curve['accuracy'].idxmax(), 'threshold']
    best_accuracy = curve['accuracy'].max()
    print(f"Best threshold: {best_threshold:.2f}, Best accuracy: {best_accuracy:.2f}")

    return chart

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB


models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVC": SVC(),
    "KNeighborsClassifier": KNeighborsClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
    "GaussianNB": GaussianNB(),
}


In [11]:
from sklearn.metrics import accuracy_score

def evaluate_features(df: pd.DataFrame, features: list[Feature], model) -> float:
    """
    Evaluate the features using the provided model.
    
    Args:
        features: List of feature identifiers
        model: The model to use for evaluation
    
    Returns:
        float: Accuracy of the model on the test set
    """
    X_train, X_test, y_train, y_test = train_test_split(
        df[[f.identifier for f in features]], df['resolved'], test_size=0.2, random_state=8675309
    )
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

def top_k_features(df: pd.DataFrame, features: list[Feature], models: dict[str, Any], k:int) -> list[Feature]:
    rows = []
    for model_id, model in models.items():
        accuracy = evaluate_features(df, features, model)
        for feature in features:
            ablated_features = [f for f in features if f != feature]
            ablated_accuracy = evaluate_features(df, ablated_features, model)
            rows.append({
                "model_id": model_id,
                "feature_id": feature.identifier,
                "accuracy": accuracy,
                "ablated_accuracy": ablated_accuracy,
            })
    accuracy_df = pd.DataFrame(rows)
    accuracy_df["importance"] = accuracy_df["accuracy"] - accuracy_df["ablated_accuracy"]
    most_important_feature_ids = accuracy_df.groupby("feature_id").agg({"importance": "mean"}).sort_values("importance", ascending=False).reset_index()["feature_id"].head(k).tolist()
    return [feature for feature in features if feature.identifier in most_important_feature_ids]
    
    
top_k_features(df, features=featurizer.features, models=models, k=15)


[Feature(identifier='parsing_regex', description='The issue involves parsing, tokenization, or regular expressions'),
 Feature(identifier='config_special_chars', description='The issue involves configuration or settings handling with special characters or syntax'),
 Feature(identifier='type_coercion', description='The issue requires understanding type coercion or handling across system boundaries'),
 Feature(identifier='cross_platform', description='The issue requires knowledge of implementation differences across platforms or versions'),
 Feature(identifier='environment_dependent', description='The issue appears in some environments but not others'),
 Feature(identifier='encoding_i18n', description='The issue involves character encoding, internationalization, or locale-specific behavior'),
 Feature(identifier='component_interactions', description='The issue requires understanding subtle interactions between components'),
 Feature(identifier='resource_optimization', description='The is

In [12]:
class Document(BaseModel):
    issue_id: str
    issue_description: str
    resolved: bool

class DocumentSplit(BaseModel):
    train: list[Document]
    test: list[Document]
    validation: list[Document]

    @staticmethod
    def initialize(data: Data, evaluation:str, test_size: float = 0.2, validation_size: float = 0.2, random_seed: int = 8675309) -> DocumentSplit:
        """Initialize a DocumentSplit object with train, test, and validation sets.

        Args:
            data: Data object containing the dataset
            evaluation: Name of the system providing the ground truth labels
            test_size: Proportion of the dataset to include in the test split
            validation_size: Proportion of the dataset to include in the validation split
            random_seed: Random seed for reproducibility
        """

        # Split the dataset into train and test sets
        train_data, test_and_validation_data = train_test_split(
            data.dataset.instances,
            test_size=test_size + validation_size,
            random_state=random_seed,
            stratify=[data.systems[evaluation].results.is_resolved(instance.instance_id) for instance in data.dataset.instances],
        )

        # Further split the test set into validation and test sets
        test_data, validation_data = train_test_split(
            test_and_validation_data,
            test_size=validation_size / (test_size + validation_size),
            random_state=random_seed,
            stratify=[data.systems[evaluation].results.is_resolved(instance.instance_id) for instance in test_and_validation_data],
        )

        # Create Document objects for each split
        train_documents = [
            Document(issue_id=instance.instance_id, issue_description=instance.problem_statement, resolved=data.systems[evaluation].results.is_resolved(instance.instance_id))
            for instance in train_data
        ]
        test_documents = [
            Document(issue_id=instance.instance_id, issue_description=instance.problem_statement, resolved=data.systems[evaluation].results.is_resolved(instance.instance_id))
            for instance in test_data
        ]
        validation_documents = [
            Document(issue_id=instance.instance_id, issue_description=instance.problem_statement, resolved=data.systems[evaluation].results.is_resolved(instance.instance_id))
            for instance in validation_data
        ]

        return DocumentSplit(train=train_documents, test=test_documents, validation=validation_documents)
    
    def to_df(self) -> pd.DataFrame:
        """Convert the DocumentSplit object to a DataFrame."""
        rows = []
        for doc in self.train:
            rows.append({
                "issue_id": doc.issue_id,
                "issue_description": doc.issue_description,
                "resolved": doc.resolved,
                "split": "train"
            })
        for doc in self.test:
            rows.append({
                "issue_id": doc.issue_id,
                "issue_description": doc.issue_description,
                "resolved": doc.resolved,
                "split": "test"
            })
        for doc in self.validation:
            rows.append({
                "issue_id": doc.issue_id,
                "issue_description": doc.issue_description,
                "resolved": doc.resolved,
                "split": "validation"
            })
        return pd.DataFrame(rows)

In [13]:
import asyncio

async def add_features(df: pd.DataFrame, features: list[Feature]) -> pd.DataFrame:
    result = df.copy()
    featurizer = Featurizer(
        identifier="baseline",
        system_prompt=system_prompt,
        message_prefix="Github issue description: ",
        features=features,
    )

    async def async_embed(issue_id: str, issue_description: str) -> FeatureEmbedding:
        # If featurizer.embed makes API calls but isn't async itself
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, featurizer.embed, issue_id, issue_description)


    issues = [
        (row["issue_id"], row["issue_description"])
        for _, row in df.iterrows()
    ]
    tasks = [
        asyncio.create_task(async_embed(issue_id, issue_description))
        for issue_id, issue_description in issues
    ]
    embeddings = await asyncio.gather(*tasks)
    embeddings_df = pd.DataFrame([embedding.to_row() for embedding in embeddings])
    result = pd.merge(result, embeddings_df, on=["issue_id"], how="left")
    return result

In [14]:
def fit_random_forest(df: pd.DataFrame, features: list[Feature]) -> RandomForestClassifier:
    # Grab just the training instances
    train_df = df[df["split"] == "train"]

    model = RandomForestClassifier(n_estimators=25, random_state=8675309)
    model.fit(train_df[[feature.identifier for feature in features]], train_df["resolved"])
    
    return model

In [15]:
def evaluate_model(df: pd.DataFrame, features: list[Feature], model: RandomForestClassifier) -> pd.DataFrame:
    scores = model.predict_proba(df[[feature.identifier for feature in features]])[:, 1]
    labels = model.predict(df[[feature.identifier for feature in features]])

    result = df.copy()
    result["score"] = scores
    result["label"] = labels

    return result

In [16]:
def validation_accuracy(df: pd.DataFrame) -> float:
    validation_df = df[df["split"] == "validation"]
    return (validation_df["resolved"] == validation_df["label"]).mean()

In [17]:
def total_cost(df: pd.DataFrame) -> float:
    return df["cost"].sum()

In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score

def abstention_performance(df: pd.DataFrame) -> dict[str, float]:
    validation_df = df[df["split"] == "validation"]
    return {
        "abstention_accuracy": validation_accuracy(df),
        "total_cost": total_cost(df),
        "precision": precision_score(validation_df["resolved"], validation_df["label"]),
        "recall": recall_score(validation_df["resolved"], validation_df["label"]),
        "f1": f1_score(validation_df["resolved"], validation_df["label"]),
    }

In [19]:
def find_score_outliers(df: pd.DataFrame, k: int = 10) -> list[Document]:
    result_df = pd.DataFrame({
        'gap': df["resolved"] - df["score"],
        'issue_id': df['issue_id'],
        'issue_description': df['issue_description'],
        'resolved': df['resolved'],
    })
    
    results = []
    for row in result_df.sort_values('gap', ascending=False).head(k).iterrows():
        doc = Document(
            issue_id=row[1]['issue_id'],
            issue_description=row[1]['issue_description'],
            resolved=row[1]['resolved'],
        )
        results.append(doc)
    return results


In [20]:
def find_low_information_features(
    df: pd.DataFrame, features: list[Feature], k: int = 5, samples: int = 10
) -> list[Feature]:
    baseline_accuracy = (
        sum(
            validation_accuracy(
                evaluate_model(
                    df,
                    features=features,
                    model=fit_random_forest(df, features=features),
                )
            )
            for _ in range(samples)
        )
        / samples
    )

    scores: dict[str, float] = {}
    for feature in features:
        ablated_features = [f for f in features if f != feature]
        accuracy = (
            sum(
                validation_accuracy(
                    evaluate_model(
                        df,
                        features=ablated_features,
                        model=fit_random_forest(df, features=ablated_features),
                    )
                )
                for _ in range(samples)
            )
            / samples
        )
        scores[feature.identifier] = baseline_accuracy - accuracy

    sorted_features = [item[0] for item in sorted(scores.items(), key=lambda x: x[1])]
    return [
        feature for feature in features if feature.identifier in sorted_features[:k]
    ]


In [21]:
def sample_features(
    current_features: list[Feature],
    forgotten_features: list[Feature],
    example_issues: list[Document],
    k: int = 1,
) -> list[Feature]:
    """
    Sample a set of features from the dataset.

    Returns:
        list[Feature]: A list of sampled features
    """
    features = []
    for _ in range(k):
        response = litellm.completion(
            **MODEL_VALUES,
            messages=[
                {
                    "role": "system",
                    "content": "You are a data science assistant that helps to identify gaps in current data representations and suggest new features. When analyzing issues and suggesting features, your goal is to create a single feature that best describe as many issues as possible while remaining focused. Good features are orthogonal to existing features, can be classified as True/False from just the issue description, have SHORT descriptions, and help to capture some complexity (or lack thereof) in the identified problems. You must use the create_feature tool to format your response."
                },
                {
                    "role": "user",
                    "content": "I need to generate a new feature that helps identify several GitHub issues. The existing features don't accurately capture the complexity (or lack thereof) of the given issues, and the forgotten features are even worse.",
                },
                {
                    "role": "user",
                    "content": f"# Existing Features\n{"\n".join([json.dumps(feature.model_dump_json()) for feature in current_features + features])}",
                },
                {
                    "role": "user",
                    "content": f"# Forgotten Features\n{"\n".join([json.dumps(feature.model_dump_json()) for feature in forgotten_features])}",
                },
                {
                    "role": "user",
                    "content": f"# Coding Issues To Address\n{"\n".join([json.dumps(issue.model_dump_json()) for issue in example_issues])}",
                },
            ],
            tools=[
                {
                    "type": "function",
                    "function": {
                    "name": "create_feature",
                    "description": "Creates a new feature with the provided identifier and description",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "identifier": {
                                "type": "string",
                                "description": "Unique identifier for the feature",
                            },
                            "description": {
                                "type": "string",
                                "description": "Detailed description of what the feature does",
                            },
                        },
                        "required": ["identifier", "description"],
                    },
                }}
                
            ],
            tool_choice={"type": "function", "function": {"name": "create_feature"}},
            temperature=0.9
        )

        feature = Feature.model_validate_json(response.choices[0].message.tool_calls[0].function.arguments)
        features.append(feature)
    return features

In [ ]:
FEATURES = negative_features + positive_features
FORGOTTEN_FEATURES = []
ITERATIONS = 10

DOCS = DocumentSplit.initialize(
    data,
    evaluation="20250203_openhands_4x_scaled",
    test_size=0.2,
    validation_size=0.2,
    random_seed=8675309
).to_df()

for i in range(ITERATIONS):

    df = await add_features(DOCS, features=FEATURES)
    model = fit_random_forest(df, features=FEATURES)
    df = evaluate_model(df, features=FEATURES, model=model)

    features_to_forget = find_low_information_features(df, features=FEATURES, k=3)

    FORGOTTEN_FEATURES.extend(features_to_forget)
    FEATURES = [feature for feature in FEATURES if feature not in FORGOTTEN_FEATURES]

    features_to_add = sample_features(
        current_features=FEATURES,
        forgotten_features=FORGOTTEN_FEATURES,
        example_issues=find_score_outliers(df, k=10),
        k=3
    )

    FEATURES.extend(features_to_add)

    with open("../data/abstention_logs.jsonl", "a") as f:
        f.write(json.dumps({
            "iteration": i,
            "features": [feature.model_dump() for feature in FEATURES],
            "forgotten_features": [feature.model_dump() for feature in FORGOTTEN_FEATURES],
            "abstention_performance": abstention_performance(df),
        }) + "\n")

Notes:

* Training loop not consistently improving performance metrics
    * Randomness in feature embedding and model training?
        * Larger random forest
        * Convert true/false labels to 0-1 labels
    * Poor feature/instance selection?
        * To boost precision, pick unabstained, unresolved instances
        * Pick features using model-based feature importance
    * Uncalibrated feature generation
* Optimization goal: boost precision, even if recall is low
* Features overlapping -- extra critic to select good subset?
* Correlation with size of fix?